# MODULES

In [1]:
!pip install pathlib
!pip install matplotlib

In [2]:
!nvidia-smi

Mon Sep 22 11:28:38 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3060        Off | 00000000:01:00.0  On |                  N/A |
|  0%   52C    P8              12W / 170W |   1243MiB / 12288MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

1.13.1+cu117
11.7
NVIDIA GeForce RTX 3060


In [4]:
import cv2
import os
import math
import time
import random
import pathlib
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image


In [5]:
class LetterboxResize:
    """
    Redimensiona a maior dimensão da imagem para target_size mantendo proporção
    e adiciona padding para formar uma imagem quadrada.
    """
    def __init__(self, target_size, fill_color=(0, 0, 0)):
        self.target_size = target_size
        self.fill_color = fill_color

    def __call__(self, img):
        # img: PIL.Image
        w, h = img.size
        scale = self.target_size / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        
        # Redimensiona proporcionalmente
        img = img.resize((new_w, new_h), resample=Image.BILINEAR)
        
        # Cria um canvas quadrado e centraliza a imagem
        new_img = Image.new("RGB", (self.target_size, self.target_size), self.fill_color)
        paste_x = (self.target_size - new_w) // 2
        paste_y = (self.target_size - new_h) // 2
        new_img.paste(img, (paste_x, paste_y))
        return new_img

In [6]:
model = YOLO("YOLO11N/train4/weights/best.pt")
model.info()  # Shows layers, params, GFLOPs, etc.

YOLO11n-cls summary: 86 layers, 1,533,666 parameters, 0 gradients, 3.3 GFLOPs


(86, 1533666, 0, 3.2543743999999997)

In [7]:
img_path = "dataset-square/val/class1/20240912_102649.jpg"
img = cv2.imread(img_path)  # BGR format by default

In [8]:
results = model(img)

probs = results[0].probs

pred_class_idx = probs.top1
pred_class_name = model.names[pred_class_idx]
pred_conf = probs.top1conf

top5_class_idx = probs.top5
top5_conf = probs.top5conf

print("Predicted class index:", pred_class_idx)
print("Predicted class name:", pred_class_name)
print("Prediction confidence:", pred_conf)
print("Top-5 class indices:", top5_class_idx)
print("Top-5 confidences:", top5_conf)


0: 1216x1216 class1 1.00, class0 0.00, 5.2ms
Speed: 27.9ms preprocess, 5.2ms inference, 0.0ms postprocess per image at shape (1, 3, 1216, 1216)
Predicted class index: 1
Predicted class name: class1
Prediction confidence: tensor(0.9961, device='cuda:0')
Top-5 class indices: [1, 0]
Top-5 confidences: tensor([0.9961, 0.0039], device='cuda:0')


In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

img_with_pred = results[0].plot()  # Returns a NumPy array (RGB) with overlay

cv2.imshow("Prediction", cv2.cvtColor(img_with_pred, cv2.COLOR_RGB2BGR))
cv2.waitKey(0)
cv2.destroyAllWindows()